# 00 · Environment setup and checks

Phase 0 of the CFFM-Net pilot: before any training, prove that every moving part works on this exact machine. If something is broken, this is the cheapest place to find out.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot` (the uploaded zip) |
| **Expected runtime** | about 10 to 15 minutes |
| **Produces** | a go / no-go answer, measured scan speed and memory, and `weights/yolo26n.pt` |

In [ ]:
# Same setup cell in every notebook: find the project, install it, check the GPU.
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        # Kaggle mounts datasets at different depths under /kaggle/input, so search five levels.
        hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
                if (Path(p).parent / "src" / "cffm").is_dir()]
        # Earlier notebooks' outputs hold stale copies of the code, so prefer the uploaded dataset.
        hits.sort(key=lambda h: any((h.parent / d).exists() for d in ("runs", "data")))
        zips = [Path(p) for k in range(1, 5) for p in glob.glob("/kaggle/input" + "/*" * k + ".zip")
                if "cffm" in Path(p).name]
        if hits:
            shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        elif zips:                       # the zip was attached but not unpacked by Kaggle
            shutil.unpack_archive(str(zips[0]), "/kaggle/working")   # it holds one folder, cffm-net-pilot/
        assert PROJECT.exists(), "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
        for p in [PROJECT, *PROJECT.rglob("*")]:     # inputs are read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle", "pytest"], check=True)   # cloudpickle: two-GPU launcher
    # Editable install, so DataLoader and DDP worker processes can import cffm too.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # also moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml

## 0. The pilot at a glance

Eleven notebooks in two phases; solid arrows carry data and checkpoints, dashed arrows metrics. The training notebooks (04, 05, 07, 08) are separate Kaggle sessions whose saved outputs 09 and 15 read.

In [ ]:
from IPython.display import Image, display
for name in ('fig_workflow',):
    f = env.project / "docs" / "figures" / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=900))
    else:
        print("missing", f)

## 1. What are we running on?

In [ ]:
import json, torch
print(json.dumps(cenv.versions(), indent=2))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GB, compute capability {p.major}.{p.minor}")
# A T4 has no bfloat16, so AMP uses float16 and cffm/scan.py keeps the recurrence in float32.
assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2' in the notebook settings."

## 2. COCO-pretrained YOLO26-n weights

Every model starts from these weights, so one copy goes in `weights/` and every run uses the identical file.

In [ ]:
import torch
w = env.weights / "yolo26n.pt"
if not w.exists():
    torch.hub.download_url_to_file("https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt", str(w))
print(w, f"{w.stat().st_size / 1e6:.1f} MB")

## 3. (Optional) the fused Mamba kernel

The pilot does **not** need it: the tested pure-PyTorch scan runs anywhere, and `mamba_ssm` takes 20+ minutes
to compile and may not support the T4. Leave the flag `False` unless you want to try; if it installs, restart
the kernel and it is picked up automatically.

In [ ]:
TRY_FUSED_SCAN = False
if TRY_FUSED_SCAN:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "causal-conv1d", "mamba-ssm"],
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    print("Now restart the kernel and run the notebook again from the top.")
from cffm import scan
print("mamba_ssm kernel:", scan.HAS_MAMBA_SSM, "| Triton kernel:", scan.HAS_TRITON)

## 4. Unit tests

Small tests, each checking one claim the method depends on. Two need the optional fused kernel and four are
an opt-in smoke run (section 7 does that run itself), so expect 6 skipped.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=env.project, capture_output=True, text=True)
print(r.stdout[-4000:])
assert r.returncode == 0, r.stderr[-4000:]

## 5. How fast is the scan here?

Forward and backward at the size of CFFM-Net's biggest scan: stride 8 on a 640 crop, so **12 800 interleaved
tokens**, **256 scan channels** and **8 images per GPU**.

In [ ]:
import time, torch
from cffm.scan import selective_scan

def time_scan(b=8, d=256, L=12800, n=8, groups=4, impl="torch", reps=3):
    dev = "cuda:0"
    u = torch.randn(b, d, L, device=dev, requires_grad=True)
    delta = torch.nn.functional.softplus(torch.randn(b, d, L, device=dev))
    A = -torch.exp(torch.randn(d, n, device=dev))
    B, C = torch.randn(b, groups, n, L, device=dev), torch.randn(b, groups, n, L, device=dev)
    D = torch.ones(d, device=dev)
    torch.cuda.reset_peak_memory_stats(); times = []
    for _ in range(reps + 1):
        torch.cuda.synchronize(); t = time.time()
        y = selective_scan(u, delta, A, B, C, D, impl=impl, **({"use_checkpoint": True} if impl == "torch" else {}))
        y.sum().backward(); torch.cuda.synchronize(); times.append(time.time() - t)
    return 1000 * sorted(times[1:])[len(times[1:]) // 2], torch.cuda.max_memory_allocated() / 2**30

for impl, ok in (("torch", True), ("triton", scan.HAS_TRITON), ("cuda", scan.HAS_MAMBA_SSM)):
    if ok:
        ms, gb = time_scan(impl=impl)
        print(f"{impl:6s} scan, stride-8 size, batch 8: {ms:7.1f} ms forward+backward, peak {gb:.2f} GB")

## 6. Do the planned batch sizes fit?

One mixed-precision forward and backward pass per model at its planned batch per GPU, measuring only memory and
step time. The last column projects one 30-epoch run on LLVIP's ~4 000 training pairs.

In [ ]:
import time, torch, pandas as pd
from ultralytics.nn.tasks import DetectionModel
from cffm.model import DualStreamDetectionModel

n_gpu = max(1, torch.cuda.device_count())
def tensors(o):
    """Every tensor inside nested dicts / lists / tuples (the head's training output is nested)."""
    if torch.is_tensor(o): return [o]
    if isinstance(o, dict): o = list(o.values())
    return [t for x in o for t in tensors(x)] if isinstance(o, (list, tuple)) else []

def step(model, b, ch, hw=(640, 640), reps=3):
    m = model.to("cuda:0").train(); opt = torch.optim.SGD(m.parameters(), lr=1e-4)
    x = torch.rand(b, ch, *hw, device="cuda:0")
    torch.cuda.reset_peak_memory_stats(); times = []
    for _ in range(reps + 1):
        torch.cuda.synchronize(); t = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = sum(t.float().mean() for t in tensors(m(x)) if t.requires_grad)   # a stand-in loss
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); times.append(time.time() - t)
    return sorted(times[1:])[len(times[1:]) // 2], torch.cuda.max_memory_allocated() / 2**30

rows, seen = [], set()
for s in (pipeline.run_spec(plan, r["name"]) for r in plan["runs"]):
    if s["model"] in seen:
        continue
    seen.add(s["model"])
    dual = s["model"].startswith("configs/")
    model = DualStreamDetectionModel(str(env.project / s["model"]), nc=1, verbose=False) if dual \
        else DetectionModel(s["model"], nc=1, verbose=False)
    per_gpu = s["batch"] // n_gpu
    t, gb = step(model, per_gpu, 4 if dual else 3)
    hours = 4000 / s["batch"] * t * s["epochs"] / 3600 * 1.15      # +15% for data loading and DDP sync
    rows.append({"model": Path(s["model"]).stem, "batch/GPU": per_gpu, "peak GB": round(gb, 2),
                 "step s": round(t, 3), "~hours for 30 ep on LLVIP": round(hours, 2)})
    del model; torch.cuda.empty_cache()
df = pd.DataFrame(rows); print(df.to_string(index=False))
limit = min(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 2**30
assert (df["peak GB"] < limit - 1.0).all(), "A batch is too large for this GPU: lower it in configs/pilot.yaml."

## 7. The whole pipeline, end to end, on fake data and both GPUs

One epoch per model on a tiny synthetic dataset, through the real trainer, two-GPU launcher, validator and
probe. The numbers mean nothing; the point is that no step crashes.

In [ ]:
from cffm.data import make_synthetic
from cffm.train import evaluate, probe, train_run

synth = make_synthetic(env.data / "synthetic", n_train=32, n_val=8, imgsz=320)
smoke = {}
for name, model, data in [("smoke_cffm", str(env.project / "configs/models/cffm-net-n.yaml"), "data_paired.yaml"),
                          ("smoke_concat", str(env.project / "configs/models/two-stream-concat-n.yaml"), "data_paired.yaml"),
                          ("smoke_visible", "yolo26n.yaml", "data_visible.yaml")]:
    best = train_run(name, model, str(synth / data), pretrained=pipeline.weights_file(env), epochs=1, imgsz=320,
                     batch=8, device=env.device, project=str(env.runs / "smoke"), workers=2, plots=False)
    smoke[name] = evaluate(best, synth / data, imgsz=320, batch=8, device=0, project=str(env.runs / "smoke_eval"))
    print(name, "ok:", {k: round(v, 3) for k, v in smoke[name].items() if k in ("mAP50(B)", "AP_s(B)")})

best = env.runs / "smoke" / "smoke_cffm" / "weights" / "best.pt"
for kind, flagged in [("clean", False), ("thermal_drop", False), ("thermal_drop", True), ("thermal_shift_8", False)]:
    m = probe(best, synth / "data_paired.yaml", kind, flagged=flagged, imgsz=320, batch=8, device=0,
              project=str(env.runs / "smoke_probe"))
    print("probe", kind, "flagged" if flagged else "", "ok")

## 8. Verdict

If every cell above ran without an error, the environment is ready. Clean up the smoke runs, then continue with
**01 · Data acquisition**.

In [ ]:
shutil.rmtree(env.runs / "smoke", ignore_errors=True)
for d in ("smoke_eval", "smoke_probe"):
    shutil.rmtree(env.runs / d, ignore_errors=True)
shutil.rmtree(env.data / "synthetic", ignore_errors=True)
print("Phase 0 checks passed. Next: 01_data_acquisition.ipynb")